# PeMS 检测站元数据分析与可视化

本notebook用于：
1. 加载并解析PeMS元数据文件
2. 统计各字段缺失值情况
3. 分析站点数量及类型分布
4. 在地图上可视化传感器位置
5. 对比多个时间点的元数据变化

## 1. 环境配置与依赖安装

In [ ]:
# 安装必要的库（如果尚未安装）
!pip install pandas folium matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import MarkerCluster
import os
import glob
import re
from datetime import datetime

# 设置中文字体（如果需要）
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("库加载完成！")

## 2. 数据加载函数定义

In [ ]:
def parse_meta_filename(filename):
    """
    从文件名解析日期信息
    例如：d03_text_meta_2024_11_27.txt -> datetime(2024, 11, 27)
    """
    pattern = r'd(\d+)_text_meta_(\d{4})_(\d{2})_(\d{2})\.txt'
    match = re.search(pattern, filename)
    if match:
        district = int(match.group(1))
        year = int(match.group(2))
        month = int(match.group(3))
        day = int(match.group(4))
        return {
            'district': district,
            'date': datetime(year, month, day),
            'date_str': f"{year}-{month:02d}-{day:02d}"
        }
    return None


def load_metadata(filepath):
    """
    加载PeMS元数据文件
    """
    # 定义列名
    columns = [
        'ID', 'Fwy', 'Dir', 'District', 'County', 'City', 
        'State_PM', 'Abs_PM', 'Latitude', 'Longitude', 'Length',
        'Type', 'Lanes', 'Name', 'User_ID_1', 'User_ID_2', 
        'User_ID_3', 'User_ID_4'
    ]
    
    df = pd.read_csv(
        filepath, 
        sep='\t', 
        names=columns,
        header=0,  # 跳过表头行
        dtype={
            'ID': str,
            'Fwy': str,
            'Dir': str,
            'District': str,
            'County': str,
            'City': str,
            'Type': str,
            'Name': str
        }
    )
    
    # 解析文件名获取日期
    file_info = parse_meta_filename(os.path.basename(filepath))
    if file_info:
        df['meta_date'] = file_info['date']
        df['meta_date_str'] = file_info['date_str']
    
    return df


print("函数定义完成！")

## 3. 配置数据路径

**请根据实际情况修改以下路径**

In [ ]:
# ============== 配置区域 ==============
# 方式1：指定包含元数据文件的目录
META_DIR = "/content/drive/MyDrive/PeMS_Data/metadata/"  # 修改为你的路径

# 方式2：直接指定要分析的文件列表
META_FILES = [
    # "/path/to/d03_text_meta_2024_11_27.txt",
    # "/path/to/d03_text_meta_2025_01_18.txt",
]

# 如果META_FILES为空，则从META_DIR自动扫描
if not META_FILES:
    META_FILES = sorted(glob.glob(os.path.join(META_DIR, "d*_text_meta_*.txt")))

print(f"找到 {len(META_FILES)} 个元数据文件：")
for f in META_FILES:
    print(f"  - {os.path.basename(f)}")

## 4. 加载单个元数据文件进行详细分析

In [ ]:
# 加载最新的元数据文件（或指定特定文件）
if META_FILES:
    current_file = META_FILES[-1]  # 使用最新文件
    print(f"正在加载: {os.path.basename(current_file)}")
    df = load_metadata(current_file)
    print(f"\n数据形状: {df.shape}")
    print(f"站点总数: {len(df)}")
else:
    print("请先配置元数据文件路径！")
    df = None

In [ ]:
# 查看数据前几行
if df is not None:
    display(df.head(10))

In [ ]:
# 查看数据基本信息
if df is not None:
    print("数据类型信息：")
    print(df.dtypes)
    print("\n" + "="*50)
    df.info()

## 5. 缺失值统计分析

In [ ]:
def analyze_missing_values(df, title="元数据缺失值分析"):
    """
    分析并可视化缺失值
    """
    # 排除辅助列
    analysis_cols = [col for col in df.columns if col not in ['meta_date', 'meta_date_str']]
    df_analysis = df[analysis_cols]
    
    # 计算缺失值统计
    missing_stats = pd.DataFrame({
        '缺失数量': df_analysis.isnull().sum(),
        '缺失比例(%)': (df_analysis.isnull().sum() / len(df_analysis) * 100).round(2),
        '非空数量': df_analysis.notnull().sum(),
        '数据类型': df_analysis.dtypes
    })
    
    # 添加空字符串统计（对于字符串列）
    empty_string_count = []
    for col in analysis_cols:
        if df_analysis[col].dtype == 'object':
            empty_count = (df_analysis[col].fillna('').astype(str).str.strip() == '').sum()
        else:
            empty_count = 0
        empty_string_count.append(empty_count)
    missing_stats['空字符串数量'] = empty_string_count
    missing_stats['有效数据比例(%)'] = ((len(df_analysis) - missing_stats['缺失数量'] - missing_stats['空字符串数量']) / len(df_analysis) * 100).round(2)
    
    print(f"\n{'='*60}")
    print(f"{title}")
    print(f"{'='*60}")
    print(f"总站点数: {len(df_analysis)}")
    print(f"\n各字段缺失情况：")
    display(missing_stats)
    
    return missing_stats


if df is not None:
    missing_stats = analyze_missing_values(df)

In [ ]:
# 可视化缺失值
if df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # 图1：缺失值数量柱状图
    cols_to_plot = missing_stats[missing_stats['缺失数量'] > 0].index.tolist()
    if cols_to_plot:
        missing_counts = missing_stats.loc[cols_to_plot, '缺失数量']
        ax1 = axes[0]
        bars = ax1.barh(cols_to_plot, missing_counts, color='coral')
        ax1.set_xlabel('Missing Count')
        ax1.set_title('Missing Values by Field')
        ax1.bar_label(bars, padding=3)
    else:
        axes[0].text(0.5, 0.5, 'No Missing Values!', ha='center', va='center', fontsize=14)
        axes[0].set_title('Missing Values by Field')
    
    # 图2：有效数据比例
    valid_ratio = missing_stats['有效数据比例(%)'].sort_values()
    colors = ['green' if v >= 90 else 'orange' if v >= 50 else 'red' for v in valid_ratio]
    ax2 = axes[1]
    bars2 = ax2.barh(valid_ratio.index, valid_ratio.values, color=colors)
    ax2.set_xlabel('Valid Data Ratio (%)')
    ax2.set_title('Valid Data Ratio by Field')
    ax2.axvline(x=90, color='green', linestyle='--', alpha=0.5, label='90%')
    ax2.axvline(x=50, color='orange', linestyle='--', alpha=0.5, label='50%')
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig('missing_values_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("图表已保存为: missing_values_analysis.png")

## 6. 站点统计分析

In [ ]:
def analyze_stations(df):
    """
    分析站点分布统计
    """
    print("\n" + "="*60)
    print("站点统计分析")
    print("="*60)
    
    # 1. 按类型统计
    print("\n【按站点类型(Type)统计】")
    type_counts = df['Type'].value_counts()
    type_mapping = {
        'ML': 'Mainline (主线)',
        'OR': 'On Ramp (入口匝道)',
        'FR': 'Off Ramp (出口匝道)',
        'HV': 'HOV Lane (HOV车道)',
        'FF': 'Fwy-Fwy Connector (高速互通)',
        'CD': 'Collector/Distributor (集散道)',
        'CH': 'Conventional Highway (常规公路)'
    }
    for t, count in type_counts.items():
        desc = type_mapping.get(t, t)
        print(f"  {t}: {count:,} 个 ({count/len(df)*100:.1f}%) - {desc}")
    
    # 2. 按方向统计
    print("\n【按行驶方向(Dir)统计】")
    dir_counts = df['Dir'].value_counts()
    for d, count in dir_counts.items():
        print(f"  {d}: {count:,} 个 ({count/len(df)*100:.1f}%)")
    
    # 3. 按高速公路统计
    print("\n【按高速公路(Fwy)统计 - Top 10】")
    fwy_counts = df['Fwy'].value_counts().head(10)
    for fwy, count in fwy_counts.items():
        print(f"  {fwy}: {count:,} 个 ({count/len(df)*100:.1f}%)")
    
    # 4. 按县统计
    print("\n【按县(County)统计】")
    county_counts = df['County'].value_counts()
    for county, count in county_counts.items():
        print(f"  County {county}: {count:,} 个 ({count/len(df)*100:.1f}%)")
    
    # 5. 车道数统计
    print("\n【按车道数(Lanes)统计】")
    lanes_counts = df['Lanes'].value_counts().sort_index()
    for lanes, count in lanes_counts.items():
        print(f"  {int(lanes) if pd.notna(lanes) else 'N/A'} 车道: {count:,} 个")
    
    return {
        'type_counts': type_counts,
        'dir_counts': dir_counts,
        'fwy_counts': fwy_counts,
        'county_counts': county_counts,
        'lanes_counts': lanes_counts
    }


if df is not None:
    station_stats = analyze_stations(df)

In [ ]:
# 可视化站点统计
if df is not None:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 图1：按类型分布
    ax1 = axes[0, 0]
    station_stats['type_counts'].plot(kind='pie', ax=ax1, autopct='%1.1f%%', startangle=90)
    ax1.set_title('Station Distribution by Type')
    ax1.set_ylabel('')
    
    # 图2：按方向分布
    ax2 = axes[0, 1]
    station_stats['dir_counts'].plot(kind='bar', ax=ax2, color=['#3498db', '#e74c3c', '#2ecc71', '#f39c12'])
    ax2.set_title('Station Distribution by Direction')
    ax2.set_xlabel('Direction')
    ax2.set_ylabel('Count')
    ax2.tick_params(axis='x', rotation=0)
    for i, v in enumerate(station_stats['dir_counts'].values):
        ax2.text(i, v + 5, str(v), ha='center')
    
    # 图3：按高速公路分布（Top 10）
    ax3 = axes[1, 0]
    station_stats['fwy_counts'].plot(kind='barh', ax=ax3, color='steelblue')
    ax3.set_title('Top 10 Freeways by Station Count')
    ax3.set_xlabel('Count')
    ax3.set_ylabel('Freeway')
    
    # 图4：车道数分布
    ax4 = axes[1, 1]
    lanes_data = station_stats['lanes_counts'].dropna()
    lanes_data.plot(kind='bar', ax=ax4, color='teal')
    ax4.set_title('Station Distribution by Lane Count')
    ax4.set_xlabel('Number of Lanes')
    ax4.set_ylabel('Count')
    ax4.tick_params(axis='x', rotation=0)
    
    plt.tight_layout()
    plt.savefig('station_statistics.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("图表已保存为: station_statistics.png")

## 7. 地图可视化 - 单个元数据文件

In [ ]:
def create_station_map(df, title="PeMS Detector Stations", use_cluster=True):
    """
    创建传感器站点地图
    
    Parameters:
    - df: 元数据DataFrame
    - title: 地图标题
    - use_cluster: 是否使用聚类标记
    """
    # 过滤有效坐标
    df_valid = df.dropna(subset=['Latitude', 'Longitude'])
    df_valid = df_valid[(df_valid['Latitude'] != 0) & (df_valid['Longitude'] != 0)]
    
    print(f"有效坐标站点数: {len(df_valid)} / {len(df)}")
    
    if len(df_valid) == 0:
        print("没有有效的坐标数据！")
        return None
    
    # 计算中心点
    center_lat = df_valid['Latitude'].mean()
    center_lon = df_valid['Longitude'].mean()
    
    # 创建地图
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=8,
        tiles='OpenStreetMap'
    )
    
    # 定义不同类型的颜色
    type_colors = {
        'ML': 'blue',      # Mainline
        'OR': 'green',     # On Ramp
        'FR': 'red',       # Off Ramp
        'HV': 'purple',    # HOV
        'FF': 'orange',    # Fwy-Fwy
        'CD': 'darkblue',  # Collector/Distributor
        'CH': 'gray'       # Conventional Highway
    }
    
    # 添加标记
    if use_cluster:
        marker_cluster = MarkerCluster(name='Stations').add_to(m)
        target = marker_cluster
    else:
        target = m
    
    for _, row in df_valid.iterrows():
        color = type_colors.get(row['Type'], 'gray')
        popup_html = f"""
        <b>Station ID:</b> {row['ID']}<br>
        <b>Name:</b> {row['Name']}<br>
        <b>Freeway:</b> {row['Fwy']} {row['Dir']}<br>
        <b>Type:</b> {row['Type']}<br>
        <b>Lanes:</b> {row['Lanes']}<br>
        <b>Length:</b> {row['Length']} mi
        """
        
        folium.CircleMarker(
            location=[row['Latitude'], row['Longitude']],
            radius=5,
            popup=folium.Popup(popup_html, max_width=300),
            color=color,
            fill=True,
            fillColor=color,
            fillOpacity=0.7
        ).add_to(target)
    
    # 添加图例
    legend_html = '''
    <div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000; 
                background-color: white; padding: 10px; border-radius: 5px;
                border: 2px solid gray; font-size: 12px;">
    <b>Station Types</b><br>
    <i style="background:blue; width:12px; height:12px; display:inline-block;"></i> ML - Mainline<br>
    <i style="background:green; width:12px; height:12px; display:inline-block;"></i> OR - On Ramp<br>
    <i style="background:red; width:12px; height:12px; display:inline-block;"></i> FR - Off Ramp<br>
    <i style="background:purple; width:12px; height:12px; display:inline-block;"></i> HV - HOV<br>
    <i style="background:orange; width:12px; height:12px; display:inline-block;"></i> FF - Fwy-Fwy<br>
    <i style="background:darkblue; width:12px; height:12px; display:inline-block;"></i> CD - Coll/Dist<br>
    <i style="background:gray; width:12px; height:12px; display:inline-block;"></i> CH - Conv Hwy
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    # 添加标题
    title_html = f'''
    <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 1000;
                background-color: white; padding: 10px; border-radius: 5px;
                border: 2px solid gray; font-size: 16px; font-weight: bold;">
    {title} (n={len(df_valid):,})
    </div>
    '''
    m.get_root().html.add_child(folium.Element(title_html))
    
    return m


print("地图函数定义完成！")

In [ ]:
# 创建单个文件的地图
if df is not None:
    date_str = df['meta_date_str'].iloc[0] if 'meta_date_str' in df.columns else 'Unknown'
    station_map = create_station_map(df, title=f"PeMS District 3 Stations ({date_str})")
    
    if station_map:
        # 保存地图
        map_filename = f"pems_stations_map_{date_str}.html"
        station_map.save(map_filename)
        print(f"地图已保存为: {map_filename}")
        
        # 在notebook中显示
        display(station_map)

## 8. 多个元数据文件对比分析

In [ ]:
def load_multiple_metadata(file_list):
    """
    加载多个元数据文件并整合
    """
    all_data = {}
    
    for filepath in file_list:
        try:
            df = load_metadata(filepath)
            file_info = parse_meta_filename(os.path.basename(filepath))
            if file_info:
                key = file_info['date_str']
                all_data[key] = df
                print(f"✓ 已加载: {os.path.basename(filepath)} ({len(df)} 站点)")
        except Exception as e:
            print(f"✗ 加载失败: {filepath} - {e}")
    
    return all_data


# 加载所有元数据文件
if META_FILES:
    all_metadata = load_multiple_metadata(META_FILES)
    print(f"\n共加载 {len(all_metadata)} 个时间点的数据")
else:
    all_metadata = {}

In [ ]:
# 多时间点站点数量变化
if all_metadata:
    print("\n" + "="*60)
    print("多时间点站点数量变化")
    print("="*60)
    
    timeline_stats = []
    for date_str, df in sorted(all_metadata.items()):
        stats = {
            'Date': date_str,
            'Total': len(df),
            'ML': len(df[df['Type'] == 'ML']),
            'OR': len(df[df['Type'] == 'OR']),
            'FR': len(df[df['Type'] == 'FR']),
            'HV': len(df[df['Type'] == 'HV']),
            'Valid_Coords': len(df.dropna(subset=['Latitude', 'Longitude']))
        }
        timeline_stats.append(stats)
    
    timeline_df = pd.DataFrame(timeline_stats)
    display(timeline_df)

In [ ]:
# 可视化时间线变化
if all_metadata and len(all_metadata) > 1:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = range(len(timeline_df))
    width = 0.15
    
    ax.bar([i - 2*width for i in x], timeline_df['ML'], width, label='ML (Mainline)', color='blue')
    ax.bar([i - width for i in x], timeline_df['OR'], width, label='OR (On Ramp)', color='green')
    ax.bar([i for i in x], timeline_df['FR'], width, label='FR (Off Ramp)', color='red')
    ax.bar([i + width for i in x], timeline_df['HV'], width, label='HV (HOV)', color='purple')
    
    ax.set_xlabel('Metadata Date')
    ax.set_ylabel('Station Count')
    ax.set_title('Station Count Changes Over Time by Type')
    ax.set_xticks(x)
    ax.set_xticklabels(timeline_df['Date'], rotation=45, ha='right')
    ax.legend()
    
    # 添加总数折线
    ax2 = ax.twinx()
    ax2.plot(x, timeline_df['Total'], 'ko-', linewidth=2, markersize=8, label='Total')
    ax2.set_ylabel('Total Stations')
    ax2.legend(loc='upper left')
    
    plt.tight_layout()
    plt.savefig('station_timeline.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("图表已保存为: station_timeline.png")

## 9. 多文件地图对比可视化

In [ ]:
def create_comparison_map(metadata_dict, dates_to_compare=None):
    """
    创建多个时间点的对比地图
    
    Parameters:
    - metadata_dict: {date_str: DataFrame} 格式的字典
    - dates_to_compare: 要对比的日期列表，None则使用所有
    """
    if dates_to_compare is None:
        dates_to_compare = sorted(metadata_dict.keys())
    
    if len(dates_to_compare) < 1:
        print("至少需要1个时间点的数据！")
        return None
    
    # 合并所有数据计算中心点
    all_lats = []
    all_lons = []
    for date in dates_to_compare:
        if date in metadata_dict:
            df = metadata_dict[date]
            valid = df.dropna(subset=['Latitude', 'Longitude'])
            all_lats.extend(valid['Latitude'].tolist())
            all_lons.extend(valid['Longitude'].tolist())
    
    center_lat = np.mean(all_lats)
    center_lon = np.mean(all_lons)
    
    # 创建地图
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=8,
        tiles='OpenStreetMap'
    )
    
    # 为每个时间点创建不同的图层
    colors = ['blue', 'red', 'green', 'purple', 'orange', 'darkblue']
    
    for i, date in enumerate(dates_to_compare):
        if date not in metadata_dict:
            continue
            
        df = metadata_dict[date]
        df_valid = df.dropna(subset=['Latitude', 'Longitude'])
        color = colors[i % len(colors)]
        
        # 创建特征组
        feature_group = folium.FeatureGroup(name=f"{date} (n={len(df_valid)})")
        
        for _, row in df_valid.iterrows():
            popup_html = f"""
            <b>Date:</b> {date}<br>
            <b>Station ID:</b> {row['ID']}<br>
            <b>Name:</b> {row['Name']}<br>
            <b>Freeway:</b> {row['Fwy']} {row['Dir']}<br>
            <b>Type:</b> {row['Type']}
            """
            
            folium.CircleMarker(
                location=[row['Latitude'], row['Longitude']],
                radius=4,
                popup=folium.Popup(popup_html, max_width=250),
                color=color,
                fill=True,
                fillColor=color,
                fillOpacity=0.6
            ).add_to(feature_group)
        
        feature_group.add_to(m)
        print(f"已添加图层: {date} ({len(df_valid)} 个站点)")
    
    # 添加图层控制
    folium.LayerControl().add_to(m)
    
    return m


print("对比地图函数定义完成！")

In [ ]:
# 创建多时间点对比地图
if all_metadata and len(all_metadata) >= 1:
    comparison_map = create_comparison_map(all_metadata)
    
    if comparison_map:
        comparison_map.save('pems_stations_comparison.html')
        print("\n对比地图已保存为: pems_stations_comparison.html")
        display(comparison_map)

## 10. 站点变化分析（新增/删除）

In [ ]:
def compare_station_changes(df_old, df_new, date_old, date_new):
    """
    比较两个时间点的站点变化
    """
    ids_old = set(df_old['ID'].astype(str))
    ids_new = set(df_new['ID'].astype(str))
    
    added = ids_new - ids_old
    removed = ids_old - ids_new
    unchanged = ids_old & ids_new
    
    print(f"\n{'='*60}")
    print(f"站点变化分析: {date_old} → {date_new}")
    print(f"{'='*60}")
    print(f"原有站点数: {len(ids_old):,}")
    print(f"现有站点数: {len(ids_new):,}")
    print(f"新增站点数: {len(added):,}")
    print(f"删除站点数: {len(removed):,}")
    print(f"未变站点数: {len(unchanged):,}")
    
    # 新增站点详情
    if added:
        print(f"\n【新增站点详情】")
        added_df = df_new[df_new['ID'].astype(str).isin(added)][['ID', 'Fwy', 'Dir', 'Type', 'Name']]
        display(added_df.head(20))
        if len(added) > 20:
            print(f"... 共 {len(added)} 个新增站点")
    
    # 删除站点详情
    if removed:
        print(f"\n【删除站点详情】")
        removed_df = df_old[df_old['ID'].astype(str).isin(removed)][['ID', 'Fwy', 'Dir', 'Type', 'Name']]
        display(removed_df.head(20))
        if len(removed) > 20:
            print(f"... 共 {len(removed)} 个删除站点")
    
    return {
        'added': added,
        'removed': removed,
        'unchanged': unchanged,
        'added_df': df_new[df_new['ID'].astype(str).isin(added)] if added else None,
        'removed_df': df_old[df_old['ID'].astype(str).isin(removed)] if removed else None
    }


# 如果有多个时间点，比较最早和最新的变化
if all_metadata and len(all_metadata) >= 2:
    dates = sorted(all_metadata.keys())
    date_old, date_new = dates[0], dates[-1]
    changes = compare_station_changes(
        all_metadata[date_old], 
        all_metadata[date_new],
        date_old, 
        date_new
    )

## 11. 导出分析报告

In [ ]:
def export_analysis_report(df, missing_stats, station_stats, output_file='pems_metadata_report.xlsx'):
    """
    导出分析报告到Excel
    """
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        # Sheet 1: 缺失值统计
        missing_stats.to_excel(writer, sheet_name='Missing Values')
        
        # Sheet 2: 类型统计
        pd.DataFrame(station_stats['type_counts']).to_excel(writer, sheet_name='Type Distribution')
        
        # Sheet 3: 高速公路统计
        pd.DataFrame(station_stats['fwy_counts']).to_excel(writer, sheet_name='Freeway Distribution')
        
        # Sheet 4: 原始数据
        df.to_excel(writer, sheet_name='Raw Data', index=False)
    
    print(f"报告已导出: {output_file}")


if df is not None:
    export_analysis_report(df, missing_stats, station_stats)

## 12. 快速使用示例

如果只想快速分析单个文件，运行以下代码：

In [ ]:
# ===== 快速分析单个文件 =====
# 取消注释并修改路径后运行

# file_path = "/path/to/your/d03_text_meta_2024_11_27.txt"
# df_quick = load_metadata(file_path)
# missing_quick = analyze_missing_values(df_quick)
# stats_quick = analyze_stations(df_quick)
# map_quick = create_station_map(df_quick)
# display(map_quick)